In [1]:
!pip install -r requirements.txt

In [2]:
!unzip data.zip

Archive:  data.zip
   creating: data/
   creating: data/audio_audacity_dist/
   creating: data/audio_audacity_dist/input/
  inflating: data/audio_audacity_dist/input/clean_signal.wav  
   creating: data/audio_audacity_dist/output/
  inflating: data/audio_audacity_dist/output/audacity_dist.wav  
   creating: data/audio_ht1/
  inflating: data/audio_ht1/guitar.wav  
   creating: data/audio_ht1/input/
  inflating: data/audio_ht1/input/ht1-input.wav  
   creating: data/audio_ht1/output/
  inflating: data/audio_ht1/output/ht1-target.wav  


In [4]:
## This script trains an LSTM according
## to the method described in
## A. Wright, E.-P. Damskägg, and V. Välimäki, ‘Real-time black-box modelling with recurrent neural networks’, in 22nd international conference on digital audio effects (DAFx-19), 2019, pp. 1–8.
import myk_data
import myk_models
import myk_loss
import myk_train
import torch
from torch.utils.data import DataLoader
import myk_evaluate
import os
import sys

from torch.utils.tensorboard import SummaryWriter
torch.backends.cudnn.benchmark = True


# used for the writing of example outputs
run_name="ssl-32-unit"
# dataset : need an input and output folder in this folder
#audio_folder = "../../data/audio_ssl/"
# this is a  dataset made with a simple audacity distortion effect
audio_folder = "/content/data/audio_audacity_dist"
assert os.path.exists(audio_folder), "Audio folder  not found. Looked for " + audio_folder
# used to render example output during training
test_file = "/content/data/audio_ht1/guitar.wav"
assert os.path.exists(test_file), "Test file not found. Looked for " + test_file

# try to read neural network size from first command-line input
lstm_hidden_size = 32

# Training configuration
learning_rate = 5e-3 # how much to adjust network by each time
seq_length_secs = 0.5 # length of audio in a sequence
batch_size = 50 # number of sequences in a batch (how parallel is it?)
max_epochs = 10000
give_up_after = 1000 # stop training if no improvement for this many epochs

# create the logger for tensorboard
expt_desc = " " + run_name + " LSTM model with " + str(lstm_hidden_size) + " hidden units"
writer = SummaryWriter(comment=expt_desc)

print("Loading dataset from folder ", audio_folder)

dataset = myk_data.generate_dataset(audio_folder + "/input/", audio_folder + "/output/", frag_len_seconds=0.5)

print("Splitting dataset")
train_ds, val_ds, test_ds = myk_data.get_train_valid_test_datasets(dataset, splits=[0.8, 0.1, 0.1])

print("Looking for GPU power")
device = myk_train.get_device()
# device = 'cpu'

print("Creating data loaders")

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=True)

print("Creating model")
model = myk_models.SimpleLSTM(hidden_size=lstm_hidden_size).to(device)

print("Creating optimiser")
# https://github.com/Alec-Wright/Automated-GuitarAmpModelling/blob/main/dist_model_recnet.py
optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, 'min', factor=0.5, patience=5, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, 'min', factor=0.5, patience=5)

print("Creating loss functions")
# https://github.com/Alec-Wright/CoreAudioML/blob/bad9469f94a2fa63a50d70ff75f5eff2208ba03f/training.py
loss_functions = myk_loss.LossWrapper()
# now the training loop

print("About to train")
lowest_val_loss = 0
best_loss = False
epochs_no_improvement = 0
model_save_dir = writer.get_logdir() + "/saved_models/"
os.makedirs(os.path.dirname(model_save_dir), exist_ok=True)

for epoch in range(max_epochs):
    # ep_loss = myk_train.train_epoch_interval(model, train_dl, loss_functions, scheduler, device=device)
    ep_loss = myk_train.train_epoch_interval(model, train_dl, loss_functions, optimiser, device=device)

    #ep_loss = myk_train.train_epoch(model, train_dl, loss_functions, optimiser, device=device)
    val_loss = myk_train.compute_batch_loss(model, val_dl, loss_functions, device=device)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Loss/train", ep_loss, epoch)

    # check if we have beaten our best loss to date
    if lowest_val_loss == 0:# first run
        lowest_val_loss = val_loss
    elif val_loss < lowest_val_loss:# new record
        lowest_val_loss = val_loss
        best_loss = True
    else:# no improvement
        best_loss = False
    if best_loss == False:
        epochs_no_improvement = epochs_no_improvement + 1
        if epochs_no_improvement > give_up_after:
            print(str(give_up_after) + " epochs with no improvement. Giving up")
            break
    else:
        epochs_no_improvement = 0

    if (best_loss) or (epoch % 250 == 0):# save best model
        print("Record loss - saving at ", epoch)
        # pytorch save
        pth_file = model_save_dir + 'lstm_size_' + str(lstm_hidden_size) + '_epoch_' + str(epoch) + "_loss_" + str(round(val_loss, 4)) + ".pth"
        torch.save(model, pth_file)
        # run some audio through the model and save as wav
        myk_evaluate.run_file_through_model(model, test_file, model_save_dir + "/" + str(epoch)+".wav", device=device)
    if best_loss: # rtneural json save is always the best yet
        model.save_for_rtneural(model_save_dir+"rtneural_model_lstm_"+str(lstm_hidden_size)+".json")

    print("epoch, train, val ", epoch, ep_loss, val_loss)




Loading dataset from folder  /content/data/audio_audacity_dist
generate_dataset:: Loaded frames from audio file 421
input tensor shape torch.Size([421, 22050, 1])
Splitting dataset
Cannot get that many items from the dataset: want 420 got 421  trimming the big one by  -1
Looking for GPU power
cuda device available
Creating data loaders
Creating model
Creating optimiser
Creating loss functions
About to train
Record loss - saving at  0
epoch, train, val  0 0.34033442 0.22918022
Record loss - saving at  1
epoch, train, val  1 0.1221608 0.097546235
Record loss - saving at  2
epoch, train, val  2 0.07302335 0.059666682
Record loss - saving at  3
epoch, train, val  3 0.06009263 0.05224232
Record loss - saving at  4
epoch, train, val  4 0.041899424 0.035390213
epoch, train, val  5 0.040357783 0.040138222
Record loss - saving at  6
epoch, train, val  6 0.032118157 0.027914355
Record loss - saving at  7
epoch, train, val  7 0.024922008 0.023791913
Record loss - saving at  8
epoch, train, val  8

In [7]:
!zip -r runs.zip /content/runs

updating: content/runs/ (stored 0%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/ (stored 0%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/ (stored 0%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/lstm_size_32_epoch_0_loss_0.2292.pth (deflated 14%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/lstm_size_32_epoch_12_loss_0.0193.pth (deflated 13%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/413.wav (deflated 4%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/lstm_size_32_epoch_4_loss_0.0354.pth (deflated 14%)
  adding: content/runs/Apr25_15-25-01_8a589a89e982 ssl-32-unit LSTM model with 32 hidden units/saved_models/lstm_size_32_epo